In [35]:
import json
import sys
from collections import defaultdict
from itertools import combinations
from pathlib import Path
import itertools
import numpy as np
import os

import pandas as pd
import torch
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns


In [36]:
MANIFOLDS = [
    "euclidean", # raw embeddings normalized to lie on the unit sphere, dimension d
    "lorentz_0.1_exp0", # raw embeddings projected to the Lorentz model using the exponential map at the origin, dimension d+1 (requires 0 padding)
    "lorentz_0.2_exp0",
    "lorentz_0.5_exp0",
    # "lorentz_1_exp0",
    # "lorentz_2_exp0",
    # "lorentz_2.5_exp0",
    # "lorentz_5_exp0",
]

In [37]:
all_dirs = [
    "../data_upload/slot_extractions_slotconstrast_dinov2",
    "../data_upload/slot_extractions_spot_coco",
    "../data_upload/slot_extractions_videosaur_dinov1",
    "../data_upload/slot_extractions_videosaur_dinov2"
]

DATA_DIR = Path("../data_upload/slot_extractions_slotconstrast_dinov2")
GT_PATH  = Path("../data_upload/gt_slotcontrast_dinov2.json")
GT_KEY   = "gt_slotcontrast_dinov2"

In [38]:
data_dir = all_dirs[0]

In [39]:
def check_is_sphere(vectors):
    return torch.allclose(vectors.norm(dim=-1), torch.ones_like(vectors.norm(dim=-1)))

def check_is_lorentz(vectors, c=1.):
    inner_product = inner_lorentz(vectors, vectors)
    return torch.allclose(inner_product, -1./c * torch.ones_like(inner_product), atol=1e-5, rtol=1e-4)


In [40]:
def exp_map0(vectors, c=torch.tensor([1.])):
    # Exponential map at the origin for the Lorentz model
    # vectors: (batch_size, dim)
    # returns: (batch_size, dim+1)
    rc_norm = torch.sqrt(c) * vectors.norm(dim=-1, keepdim=True)
    time_component_at_origin = torch.sqrt(1./c)
    time_component = time_component_at_origin * torch.cosh(rc_norm)
    spacial_component = torch.sinh(rc_norm) * vectors / torch.clamp(rc_norm, min=1e-5)
    return torch.cat([time_component, spacial_component], dim=-1)
    
def inner_lorentz(x, y):
    # Inner product in the Lorentz model
    # x, y: (batch_size, dim+1)
    # returns: (batch_size,)
    return -x[..., 0] * y[..., 0] + torch.sum(x[..., 1:] * y[..., 1:], dim=-1)

def norm_lorentz(x, tol=1e-5):
    # Norm in the Lorentz model
    # x: (batch_size, dim+1)
    # returns: (batch_size,)
    return torch.sqrt(torch.clamp(torch.abs(inner_lorentz(x, x)), min=tol))


def projx_lorentz(x, c=1.):
    # Projection to the Lorentz model
    # x: (batch_size, dim)
    # returns: (batch_size, dim+1)
    rc_norm = torch.sqrt(c) * x.norm(dim=-1, keepdim=True)
    time_component = torch.sqrt(1./c + rc_norm**2)
    return torch.cat([time_component, x], dim=-1)


def get_origin_lorentz(dim, c=torch.tensor([1.]), dtype=torch.float32):
    time_component_at_origin = torch.sqrt(1./c)
    return torch.cat([time_component_at_origin * torch.ones(1, dtype=dtype), torch.zeros(dim-1, dtype=dtype)], dim=0)


def proj_to_tgz_lorentz(x, z, c=1.):
    # Projection of a point in space to the tangent space at the origin for the Lorentz model
    # x: (batch_size, dim)
    # returns: (batch_size, dim)
    return x + c * z * inner_lorentz(x, z).unsqueeze(-1)


def centroid_lorentz(points, c=1., tol=1e-5):
    # Compute the centroid of a set of points in the Lorentz model using standard mean
    # points: (num_points, dim+1)
    # returns: (dim+1,)
    mean = points.mean(dim=0)
    return mean / (norm_lorentz(mean, tol=tol) * torch.sqrt(c))


def distance_lorentz(x, y, c=1.0, eps=1e-7):
    # Distance in the Lorentz model
    # x, y: (batch_size, dim+1)
    inner_prod = inner_lorentz(x, y)
    z = -inner_prod * c
    z = torch.clamp(z, min=1.0 + eps)
    return torch.acosh(z) / torch.sqrt(torch.tensor(c, dtype=x.dtype, device=x.device))


def distance_norm_lorentz(x, y, c=1.):
    # Distance in the Lorentz model using the norm of the difference
    # x, y: (batch_size, dim+1)
    # returns: (batch_size,)
    diff = x - y
    return norm_lorentz(diff)

In [41]:
def stereographic_proj(x):
    # Stereographic projection to the sphere
    # x: (batch_size, dim)
    # returns: (batch_size, dim+1)
    norm_sq = x.norm(dim=-1, keepdim=True)**2
    time_component = (norm_sq - 1) / (norm_sq + 1)
    spacial_component = 2 * x / (norm_sq + 1)
    return torch.cat([time_component, spacial_component], dim=-1)

def distance_sphere(x, y):
    # Distance on the sphere with cosine similarity
    # x, y: (batch_size, dim+1)
    # returns: (batch_size,)
    inner_prod = torch.sum(x * y, dim=-1)
    return torch.acos(inner_prod.clamp(-1 + 1e-7, 1 - 1e-7))

In [42]:
def pairwise_dist(vectors, manifold_name):
    """Compute (N, N) geodesic distance matrix.

    Args:
        vectors: (N, D)
        manifold_name: one of MANIFOLDS.

    Returns:
        (N, N) distance tensor.
    """
    if manifold_name.startswith("lorentz"):
        c = float(manifold_name.split("_")[1])
        # Lorentz inner: -x0*y0 + x_rest @ y_rest^T → (P, C)
        inner = -vectors[:, 0:1] @ vectors[:, 0:1].T + vectors[:, 1:] @ vectors[:, 1:].T
        return torch.acosh((-inner * c).clamp(min=1.0)) / np.sqrt(c)
    else:
        norms = vectors.norm(dim=1, keepdim=True)
        cos_sim = vectors @ vectors.T / (norms @ norms.T)
        return torch.acos(cos_sim.clamp(-1 + 1e-7, 1 - 1e-7))

def distance(x, y, manifold_name, c=None):
    if manifold_name.startswith("lorentz"):
        c = torch.tensor(float(manifold_name.split("_")[1]), dtype=vectors.dtype)
        return distance_lorentz(x, y, c)
    else:
        return distance_sphere(x / x.norm(dim = 1, keepdim=True), y / y.norm())

In [43]:
def compute_centroid(vectors, manifold_name):
    """Manifold-appropriate centroid for the synthetic root.

    Args:
        vectors: (N, D) tensor on the manifold.
        manifold_name: one of MANIFOLDS.

    Returns:
        (D,) centroid tensor.
    """
    if manifold_name.startswith("lorentz"):
        c = torch.tensor(float(manifold_name.split("_")[1]), dtype=vectors.dtype)
        return centroid_lorentz(vectors, c=c)
    else:
        return vectors.mean(dim=0)

In [44]:
def get_origin(dim, manifold_name):
    if manifold_name.startswith("lorentz"):
        c = torch.tensor(float(manifold_name.split("_")[1]))
        return get_origin_lorentz(dim, c=c)
    else:
        return torch.zeros(dim)

In [45]:
def project_to_manifold(vectors, manifold):
    vectors = vectors.to(torch.float64)
    if manifold == "euclidean":
        return vectors
    elif manifold == "sphere":
        vectors = vectors / torch.norm(vectors, dim=-1, keepdim=True)
        assert check_is_sphere(vectors), "Projection to sphere failed, not all vectors have norm 1"
        return vectors
    elif manifold.endswith("_exp0"):
        c = torch.tensor(float(manifold.split("_")[1]), dtype=vectors.dtype)
        vectors = exp_map0(vectors, c=c)
        assert check_is_lorentz(vectors, c=c), "Projection to Lorentz failed, not all vectors satisfy the Lorentz condition"
        return vectors
    elif manifold == "lorentz_1_projx":
        c = torch.tensor(float(manifold.split("_")[1]), dtype=vectors.dtype)
        vectors = projx_lorentz(vectors, c=c)
        assert check_is_lorentz(vectors, c=c), "Projection to Lorentz failed, not all vectors satisfy the Lorentz condition"
        return vectors
    elif manifold == "sphere_estereographic":
        vectors = stereographic_proj(vectors)
        assert check_is_sphere(vectors), "Projection to sphere failed, not all vectors have norm 1"
        return vectors
    elif manifold == "lorentz_1_exp0_restricted":
        c = torch.tensor(float(manifold.split("_")[1]), dtype=vectors.dtype)
        origin = get_origin_lorentz(vectors.shape[-1], c=c, dtype=vectors.dtype)
        proj_vectors = proj_to_tgz_lorentz(vectors, origin, c=c)
        assert torch.all(proj_vectors[:, 0] == 0), "Projection to tangent space slot_extractions_videosaur_dinov2at origin failed, time component is not 0"
        vectors = exp_map0(proj_vectors[:, 1:], c=c)
        assert check_is_lorentz(vectors, c=c), "Projection to Lorentz failed, not all vectors satisfy the Lorentz condition"
        return vectors
    elif manifold == "lorentz_1_projx_restricted":
        c = torch.tensor(float(manifold.split("_")[1]), dtype=vectors.dtype)
        vectors = projx_lorentz(vectors[:, 1:], c=c)
        assert check_is_lorentz(vectors, c=c), "Projection to Lorentz failed, not all vectors satisfy the Lorentz condition"
        return vectors

        
    else: 
        raise ValueError(f"Unknown manifold: {manifold}")
        

        

In [46]:
# for data_dir in all_dirs:
print(f"Processing directory: {data_dir}")
data_dir = Path(data_dir)

video_dirs = sorted(d for d in data_dir.iterdir() if d.is_dir() and (d.name.startswith("video_") or d.name.startswith("image_")))
print(f"Found {len(video_dirs)} instances in {data_dir}")

embeddings_info = defaultdict(dict)

for video_dir in tqdm(video_dirs, desc=f"Processing videos in {data_dir.name}"):
    # Take files of form slots_raw_*.pt
    all_slots = list(video_dir.glob("slots_raw_*.pt"))
    embeddings_info[video_dir.name] = defaultdict(dict)
    for slots in all_slots:
        # embeddings_info[video_dir.name][slots.stem] = defaultdict(dict)
        with open(slots, "rb") as f:
            data = torch.load(f)
            # Only keep the last frame if it is not coco
            if "coco" not in data_dir.stem:
                embeddings_info[video_dir.name]["raw"][slots.stem.split("_")[-1]] = data[-1]
            else:
                embeddings_info[video_dir.name]["raw"][slots.stem.split("_")[-1]] = data
        for manifold in MANIFOLDS:
            projected = project_to_manifold(
                embeddings_info[video_dir.name]["raw"][slots.stem.split("_")[-1]].to(torch.float64),
                manifold,
            )
            embeddings_info[video_dir.name][f"{manifold}"][slots.stem.split("_")[-1]] = projected


# Save the embeddings info to a .pt file
print(f"Saving embeddings info for {data_dir}...")
output_file = data_dir / "embeddings_info.pt"
with open(output_file, "wb") as f:
    torch.save(embeddings_info, f)

Processing directory: ../data_upload/slot_extractions_slotconstrast_dinov2
Found 300 instances in ../data_upload/slot_extractions_slotconstrast_dinov2


Processing videos in slot_extractions_slotconstrast_dinov2: 100%|██████████| 300/300 [00:01<00:00, 251.65it/s]


Saving embeddings info for ../data_upload/slot_extractions_slotconstrast_dinov2...


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Hit@1 and Hit@3 — Parent Retrieval Evaluation
# For each fine slot s_j: rank all coarse slots by distance, check if GT parent
# is rank 1 (hit@1) or in top 3 (hit@3). Evaluated for all manifolds.
# ═══════════════════════════════════════════════════════════════════════════════

import json
import gc
import numpy as np
import torch
import pandas as pd
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data_upload/slot_extractions_slotconstrast_dinov2")
GT_PATH  = Path("../data_upload/gt_slotcontrast_dinov2.json")
GT_KEY   = "gt_slotcontrast_dinov2"

MANIFOLDS_EVAL = [
    "euclidean",
    "lorentz_0.1_exp0",
    "lorentz_0.2_exp0",
    "lorentz_0.5_exp0",
    "lorentz_1_exp0",
    "lorentz_2_exp0",
]
GT_TRANSITIONS = [(3, 5), (5, 7), (7, 11), (11, 13)]  # adjacent levels only
# ─────────────────────────────────────────────────────────────────────────────

# Load GT and build: gt_assign[video][(p_lvl, c_lvl)][child_idx] = parent_idx
with open(GT_PATH) as f:
    gt_raw = json.load(f)[GT_KEY]

gt_assign = {}
for video_name, video_gt in gt_raw.items():
    gt_assign[video_name] = {}
    for p_lvl, c_lvl in GT_TRANSITIONS:
        key = f"{p_lvl}_to_{c_lvl}"
        if key not in video_gt:
            continue
        assign = {}
        for parent_label, children in video_gt[key].items():
            p_idx = int(parent_label.split("_")[1])
            for child_label in children:
                c_idx = int(child_label.split("_")[1])
                assign[c_idx] = p_idx
        gt_assign[video_name][(p_lvl, c_lvl)] = assign
del gt_raw


def cross_dist(child_vecs, parent_vecs, manifold_name):
    """(C, P) distance matrix between child and parent slot vectors."""
    if manifold_name.startswith("lorentz"):
        c = float(manifold_name.split("_")[1])
        inner = (-child_vecs[:, 0:1] * parent_vecs[:, 0:1].T
                 + child_vecs[:, 1:] @ parent_vecs[:, 1:].T)
        return torch.acosh((-inner * c).clamp(min=1.0)) / np.sqrt(c)
    else:  # euclidean: geodesic on unit sphere = arccos of cosine similarity
        cn = child_vecs  / child_vecs.norm(dim=1, keepdim=True).clamp(min=1e-8)
        pn = parent_vecs / parent_vecs.norm(dim=1, keepdim=True).clamp(min=1e-8)
        return torch.acos((cn @ pn.T).clamp(-1 + 1e-7, 1 - 1e-7))


# Load embeddings and run retrieval
print("Loading embeddings_info...")
embeddings_info = torch.load(DATA_DIR / "embeddings_info.pt", weights_only=False)

hits = defaultdict(lambda: defaultdict(lambda: {"h1": [], "h3": []}))

for video_name, video_data in tqdm(embeddings_info.items(), desc="Evaluating Hit@k"):
    if video_name not in gt_assign:
        continue
    for manifold_name in MANIFOLDS_EVAL:
        if manifold_name not in video_data:
            continue
        mdata = video_data[manifold_name]
        for (p_lvl, c_lvl), assign in gt_assign[video_name].items():
            if not assign:
                continue
            p_key, c_key = str(p_lvl), str(c_lvl)
            if p_key not in mdata or c_key not in mdata:
                continue
            pvecs  = mdata[p_key].to(torch.float64)          # (P, D)
            cvecs  = mdata[c_key].to(torch.float64)          # (C, D)
            dists  = cross_dist(cvecs, pvecs, manifold_name) # (C, P)
            ranked = dists.argsort(dim=1)                    # ascending: nearest first

            for c_idx, p_idx_gt in assign.items():
                if c_idx >= cvecs.shape[0] or p_idx_gt >= pvecs.shape[0]:
                    continue
                rank = (ranked[c_idx] == p_idx_gt).nonzero(as_tuple=True)[0].item()
                hits[(p_lvl, c_lvl)][manifold_name]["h1"].append(int(rank == 0))
                hits[(p_lvl, c_lvl)][manifold_name]["h3"].append(int(rank < 3))

del embeddings_info
gc.collect()

# ── Results table ─────────────────────────────────────────────────────────────
rows = []
for (p_lvl, c_lvl) in GT_TRANSITIONS:
    for manifold_name in MANIFOLDS_EVAL:
        r = hits[(p_lvl, c_lvl)][manifold_name]
        if not r["h1"]:
            continue
        n  = len(r["h1"])
        h1 = 100 * sum(r["h1"]) / n
        h3 = 100 * sum(r["h3"]) / n
        rows.append({
            "Transition": f"L{p_lvl}→L{c_lvl}",
            "Manifold":   manifold_name,
            "Hit@1 (%)":  round(h1, 1),
            "Hit@3 (%)":  round(h3, 1),
            "N":          n,
        })

df = pd.DataFrame(rows)
print("\n", df.pivot_table(
    index="Transition", columns="Manifold",
    values=["Hit@1 (%)", "Hit@3 (%)"]
).to_string())